# Introspecting a Datasource and a Dataset via MCP

SLayer's MCP (Model Context Protocol) server exposes tools like `query`, `datasets_summary`, `create_dataset`, and — the focus of this notebook — **`describe_datasource`** and **`inspect`**, which together tell an LLM agent everything it needs to know before issuing a query.

`inspect` with `entity_type="dataset"` and `compact=False` returns a complete markdown view of a dataset: metadata, all filters, fields, metrics, custom aggregations, direct relationships, and a data-profile table run against the live database. The Fields table gets a `sampled` column that carries the per-field data profile inline — distinct values for string/boolean fields, `min .. max` for number/date/time fields. Every table auto-prunes columns that are empty for every row and collapses to a comma-separated backticked list when only one column survives.

The parameters used here:

| Parameter | Default | Effect |
|-----------|---------|--------|
| `reference` | — | The dataset to inspect (a bare name or `<datasource>.<dataset>`) |
| `entity_type` | — required — | `"dataset"` here; also `datasource`, `field`, `metric`, `aggregation`, `memory` |
| `compact` | `true` | `true` gives a cheap schema skeleton (names only, no database calls); `false` the full render |
| `num_rows` | `3` | Max rows in the data-profile table |

### A note on transport

`slayer mcp` runs over **stdio**, not HTTP, so a literal `curl` can't reach it. In this notebook we invoke each tool through the **in-process Python MCP client** — `await mcp.call_tool(name=..., arguments=...)` — which issues the same JSON-RPC `tools/call` request a desktop MCP client would issue over stdio. The REST API (`slayer serve` / port 5143) is a separate surface and is not used here.

We pick **`items`** as the subject of `inspect` because it has a rich relationship graph in Jaffle Shop: direct relationships to `orders` and `products`, plus multi-hop reach to `customers` and `stores` through `orders`.

See also: [Relationships](../05_relationships/relationships.md) · [Auto Ingestion](../03_auto_ingest) · [Multi-Stage Queries](../06_multistage_queries)

In [1]:
import json
import os
import sys
import warnings
from pathlib import Path

# The mcp SDK emits a benign PydanticJsonSchemaWarning when it builds the tool
# JSON schemas; silence it so the tool listing below stays clean.
warnings.filterwarnings("ignore", message=".*not JSON serializable.*")

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, datasets = ensure_jaffle_shop()
print(f"Loaded {len(datasets)} datasets: {sorted(d.name for d in datasets)}")

Loaded 8 datasets: ['customers', 'items', 'orders', 'products', 'regions', 'stores', 'supplies', 'tweets']


## Why `items`?

During ingestion SLayer creates one `DatasetRelationship` per FK on the source table. `items` has direct FKs to `orders` and `products`. Multi-hop targets like `customers` and `stores` are reached at query time by walking through `orders`'s own relationships:

```
items ──► orders ──► customers
      │         └──► stores
      └──► products
```

The next cell prints the raw `DatasetRelationship` entries on `items` and `orders`.

In [2]:
items = next(d for d in datasets if d.name == "items")
orders_d = next(d for d in datasets if d.name == "orders")

print("items relationships (direct FKs only):")
for j in items.relationships:
    src, tgt = j.column_pairs[0]
    print(f"  -> {j.to:<10} ON {src:<22} = {tgt}")

print("\norders relationships (direct FKs only):")
for j in orders_d.relationships:
    src, tgt = j.column_pairs[0]
    print(f"  -> {j.to:<10} ON {src:<22} = {tgt}")

print("\nMulti-hop: items reaches customers/stores via orders.relationships")

items relationships (direct FKs only):
  -> orders     ON order_id               = id
  -> products   ON sku                    = sku

orders relationships (direct FKs only):
  -> customers  ON customer_id            = id
  -> stores     ON store_id               = id

Multi-hop: items reaches customers/stores via orders.relationships


## Wiring up the MCP client

We instantiate the **MCP** server (from `slayer.mcp.server`, _not_ the REST app in `slayer.api.server`) against the same storage the engine just loaded. `create_mcp_server` returns an mcp `MCPServer`; its `list_tools()` and `call_tool(name, arguments)` methods are exactly what stdio clients invoke under the hood.

Over stdio, each `call_tool` request serialises to a JSON-RPC frame like this (the `params` payload is what would be the POST body if SLayer's MCP ran on HTTP):

```json
{
  "jsonrpc": "2.0", "id": 1, "method": "tools/call",
  "params": {
    "name": "inspect",
    "arguments": {"reference": "items", "entity_type": "dataset", "compact": false, "num_rows": 10}
  }
}
```

In [3]:
from slayer.mcp.server import create_mcp_server

mcp = create_mcp_server(storage=storage)

tools = await mcp.list_tools()
print(f"MCP server exposes {len(tools)} tools:")
for t in tools:
    marker = "<-- targets" if t.name in ("describe_datasource", "inspect") else ""
    print(f"  - {t.name} {marker}")

inspect_tool = next(t for t in tools if t.name == "inspect")
print("\ninspect input schema:")
print(json.dumps(inspect_tool.input_schema, indent=2))

MCP server exposes 22 tools:
  - query 
  - datasets_summary 
  - inspect <-- targets
  - create_dataset 
  - edit_dataset 
  - create_datasource 
  - list_datasources 
  - describe_datasource <-- targets
  - edit_datasource 
  - delete_dataset 
  - create_semantic_model 
  - edit_semantic_model 
  - delete_semantic_model 
  - validate_datasets 
  - recommend_population 
  - delete_datasource 
  - ingest_datasource 
  - set_datasource_priority 
  - get_datasource_priority 
  - save_memory 
  - forget_memory 
  - search 

inspect input schema:
{
  "properties": {
    "entity_type": {
      "description": "Required: datasource, dataset, field, metric, aggregation, memory or semantic_model. Asserts the resolved kind and disambiguates a name shared by, e.g., a field and an aggregation.",
      "title": "Entity Type",
      "type": "string"
    },
    "reference": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "items": {
            "type": "string"
 

## Step 1 — Introspect the datasource

Before drilling into a specific dataset, a well-behaved agent verifies the datasource: is it reachable, what's its type, what schemas are available, and what tables live there? The MCP `describe_datasource` tool covers all of that in one call. Pass `list_tables=False` to skip the table-listing block if you just want the connection probe; pass `schema_name="..."` to scope the table listing.

The Jaffle Shop fixture registered a DuckDB datasource named `jaffle_shop` when we called `ensure_jaffle_shop()`, so let's inspect it.

In [4]:
describe_args = {"name": "jaffle_shop"}
print("Arguments (the 'curl POST body' equivalent):")
print(json.dumps(describe_args, indent=2))

content = (await mcp.call_tool(name="describe_datasource", arguments=describe_args)).content
# Show any absolute DB path relative to the repo root so the committed output
# carries no local username / workstation path.
repo_root = Path.cwd().parents[2]
print("\n=== describe_datasource response ===\n")
print(content[0].text.replace(f"{repo_root}/", ""))

Arguments (the 'curl POST body' equivalent):
{
  "name": "jaffle_shop"
}

=== describe_datasource response ===

Datasource: jaffle_shop
Type: duckdb
Database: docs/examples/jaffle_data/demo/jaffle_shop.duckdb

Connection: OK
Available schemas: jaffle_shop.main, system.information_schema, system.main, temp.main

Tables (8):
  - customers
  - items
  - orders
  - products
  - regions
  - stores
  - supplies
  - tweets

Use ingest_datasource to create datasets from these tables.


## Step 2 — Survey the datasource's datasets with `datasets_summary`

Now that we know the datasource works, `datasets_summary` gives a compact markdown overview of every dataset registered against it: name, description, its **field count**, its **metric names**, and the datasets it has relationships to. It's deliberately lighter than `inspect` with `compact=False` — no types, no distinct values, no sample data — so an agent can scan N datasets without making N heavy calls.

In [5]:
summary_args = {"datasource_name": "jaffle_shop"}
print("Arguments (the 'curl POST body' equivalent):")
print(json.dumps(summary_args, indent=2))

content = (await mcp.call_tool(name="datasets_summary", arguments=summary_args)).content
print("\n=== datasets_summary response (markdown) ===\n")
print(content[0].text)

Arguments (the 'curl POST body' equivalent):
{
  "datasource_name": "jaffle_shop"
}

=== datasets_summary response (markdown) ===

# Datasource: `jaffle_shop` — 9 dataset(s)

## `customers`
Customers of the Jaffle Shop — one row per person.
Fields: 2
Metrics: customer_count
Relationships to: `orders`, `tweets`

## `items`
Order line items — one row per unit sold on an order.
Fields: 3
Metrics: units_sold
Relationships to: `orders`, `products`

## `orders`
Customer orders — one row per order. Monetary amounts are in dollars. The fact table at the center of the demo: has relationships to customers and stores, and is referenced by items.
Fields: 9
Metrics: total_revenue, net_sales, tax_collected, order_count, unique_customers, avg_order_value, effective_tax_rate, sales_weighted_aov
Relationships to: `customers`, `item_skus`, `items`, `stores`

## `products`
Menu items sold at the Jaffle Shop — jaffles and beverages.
Fields: 6
Metrics: product_count, avg_price
Relationships to: `ingredient

## Step 3 — Drill into one dataset with `inspect`

We've picked `items` from the `datasets_summary` output above. `inspect` with `compact=False` returns a single markdown document — we print it verbatim rather than parsing as JSON.

In [6]:
arguments = {"reference": "items", "entity_type": "dataset", "compact": False, "num_rows": 10}
print("Arguments (the 'curl POST body' equivalent):")
print(json.dumps(arguments, indent=2))

content = (await mcp.call_tool(name="inspect", arguments=arguments)).content
markdown = content[0].text

print(f"\nResponse length: {len(markdown):,} characters")
print("\n=== inspect response (markdown) ===\n")
print(markdown)

Arguments (the 'curl POST body' equivalent):
{
  "reference": "items",
  "entity_type": "dataset",
  "compact": false,
  "num_rows": 10
}

Response length: 2,282 characters

=== inspect response (markdown) ===

# Dataset: `items`

Order line items — one row per unit sold on an order.

- **data_source:** `jaffle_shop`
- **sql_table:** `items`
- **row_count:** 982,197

## Fields (3)

| name | datatype | primary_key | allowed_aggregations | label | description | sampled |
| --- | --- | --- | --- | --- | --- | --- |
| id | String | yes | all | Item ID | — | — |
| order_id | String | — | all | Order | Order this line belongs to. | 00021b23-77a8-4aa1-8dec-ce26ee97d801, 000ca8e2-8cf8-4478-9ee3-8da5c1c37054, 0020d5fe-c9c0-4c56-95ee-438c705b708f, 005366f2-7cb5-4e7b-9f63-dbe832e34038, 00651419-7e56-482c-980c-186a6195d6a4, 0089a0d7-3394-4a1e-8ad6-4bdb59e6bddf, 009578a7-d178-4470-adfd-1de169d8fe4f, 00985775-8c1a-4b30-b3be-cb2c7bd88147, 00a528ac-5147-4f38-9b21-6d220e3b09e3, 00b7450d-8719-4204-8a9e-

## What we just got

The exploration flow was `describe_datasource` → `datasets_summary` → `inspect` — exactly the sequence an MCP-driven agent would run before composing its first `query` call.

`describe_datasource` gave us the datasource type (`duckdb`), the file backing it, a successful connection probe, the available schemas, and the list of tables in one call. `datasets_summary` returned a brief overview of every dataset in that datasource. `inspect` returned a single markdown document covering:

- `# Dataset: <name>` header and description
- Metadata bullets: `data_source`, `sql_table`, `default_time_dimension`, `row_count`
- `## SQL` fenced block when the dataset has a custom SQL expression (query-backed datasets)
- `## Filters (dataset-level)` — always-applied WHERE conditions
- `## Fields` — markdown table (`name`, `datatype`, `primary_key`, `allowed_aggregations`, `label`, `description`) with a `sampled` column that embeds the per-field data profile: distinct values (e.g. the SKUs for `sku`) for string/boolean fields, and `<min> .. <max>` for number/date/time fields
- `## Metrics` — markdown table with `name`, `expression`, `label`, `description`
- `## Aggregations` — custom `Aggregation` definitions with their expressions
- `## Relationships` — target dataset, key pairs, and cardinality
- `## Data Profile` — a real `SlayerQuery` executed through `engine.execute`, with `count(<dataset>.*)` plus one aggregation per field (`avg` when allowed, otherwise the first permitted aggregation)

Every markdown table in the response auto-prunes empty columns. When exactly one column survives pruning, the table collapses to a comma-separated backticked list — which is why a field or metric table from `inspect` on an auto-ingested dataset (where descriptions are often absent) often renders as a dense list of names rather than a two-column table of mostly em-dashes.

Other MCP tools that complement these for an exploring agent: `list_datasources` (enumerate datasources) and `datasets_summary(datasource_name="...")` — the latter returns a compact markdown summary of every dataset in one datasource (name + description + field count + metric names + list of related dataset names), much lighter than `inspect` and great for picking the right dataset to drill into next.